# K5 trained on T2 (shown as DADA2000): per-video curves on `nexar_val`, whole videos

Illustration only (s2024, n = 1), **decides nothing** (`core/docs/final/ZS_CURVES.md`, "Nexar val"). §13 of `KIM_RUN.md` read these two rows on `nexar_val` but kept no per-frame scores. This notebook scores them again with `--save-scores` into a **new** folder, checks that the numbers reproduce §13, and draws the curves. **Caveat:** on Nexar the event sits mid-video. A ruler that only measures distance to the middle scores 0.9735 whole-video macro, above K5-T2's 0.8201, so these curves show shape only. `nexar_test` is never touched (`--final` is never passed).

Upload branch `v_final_las_vad` to `Thesis-Final/kat-vad` first (it needs the `nexar` curve set).

| step | what | GPU | estimate |
|---|---|---|---|
| 1-3 | setup, preflight | no | 2 min |
| 4 | `nexar_eval --save-scores`: LaGoVAD, K5 (T2) -> `outputs/kim_t2/eval/nexar_curves/<row>` + reproduction check vs §13 | yes | 15-30 min |
| 5 | `zs_curves`: grid + onset + one PNG per video -> `outputs/kim_t2/REPORTS/figures/train_t2/nexar/` | no | 2-3 min |

A re-run skips what is on Drive. Thesis and Thesis-V2 are only read; everything new goes to Thesis-Final.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read-only: data, ckpts/best.ckpt
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # read-only: Nexar caches, census, T2 statistics
DRIVE_F  = '/content/drive/MyDrive/Thesis-Final'  # code, K5 (T2); every output of this notebook
os.environ['PROJECT_ROOT']       = DRIVE_F
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE_F}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_F}/outputs'

os.environ['REPO'] = f'{DRIVE_F}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v_final_las_vad there'
assert 'ZS_CURVES_NEXAR_SET' in (REPO / 'core/constants.py').read_text(), 'this upload has no nexar curve set -- re-upload kat-vad/'
assert '--save-scores' in (REPO / 'core/tools/nexar_eval.py').read_text(), 'nexar_eval has no --save-scores -- re-upload kat-vad/'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'

K5T2, LAG = constants.KIM_T2_ROW, constants.ZS_LAGOVAD_ARM
SEED     = constants.KIM_SEED                      # 2024
ENC      = constants.V5_ENCODER                    # V2-S
STEPS    = constants.NEXAR_STEP_BUDGET             # 1,740: K5 (T2)'s J10 step
NEXAR_SPLIT = constants.V2_SPLIT_NEXAR_VAL         # never nexar_test
NX       = constants.ZS_CURVES_NEXAR_SET
MAIN_LABEL = f"{constants.ZS_CURVES_MAIN_LABEL} (trained on {constants.ZS_CURVES_TRAIN_LABELS['t2']})"
REPRO_TOL = 1e-4                                   # |new - §13| allowed (GPU kernels are not bit-stable)

V2_CACHE, V2_OUT = Path(DRIVE_V2) / 'cache', Path(DRIVE_V2) / 'outputs'
A2_T2    = V2_CACHE / 'v2' / 'A2_S' / constants.DADA_ORIGIN_DATASET   # K5 (T2)'s statistics
BEST     = constants.CKPT_ROOT / 'best.ckpt'
CENSUS   = V2_OUT / 'v2' / 'REPORTS' / 'nexar_n0' / 'census.json'
NEXAR_X  = V2_CACHE / 'clip' / constants.NEXAR_CLIP_S1_DATASET
NEXAR_U  = V2_CACHE / 'video' / ENC / f'{constants.NEXAR_VIDEO_S1_DATASET}_squash'
NEXAR_DEFS = Path(DRIVE_V2) / 'data' / f'{constants.NEXAR_DATASET}_{constants.NEXAR_CONSTRUCTION_WINDOW}'

ROOT     = constants.OUTPUT_ROOT / constants.KIM_T2_OUTPUT_DIRNAME
RUN      = ROOT / constants.KIM_MAIN_ARM / f's{SEED}'
CKPT     = RUN / 'checkpoint_last.pt'
OLD      = ROOT / 'eval' / 'nexar'                                    # §13's read-outs (no scores)
NEW      = ROOT / 'eval' / constants.ZS_CURVES_NEXAR_DIRNAME          # this notebook: + scores/
TAG      = constants.ZS_CURVES_DEFAULT_TAG
FIGS     = ROOT / 'REPORTS' / constants.ZS_CURVES_FIGURES_DIRNAME / 'train_t2' / NX / TAG
LOCAL    = Path('/content/k5_t2_nexar_curves')
LOCAL.mkdir(parents=True, exist_ok=True)

INPUTS = (('K5 (T2) checkpoint', CKPT), ('K5 (T2) verified', RUN / 'verified.json'),
          ('T2 A2 stats', A2_T2 / constants.V2_INPUT_MANIFEST_FILENAME), ('best.ckpt', BEST),
          ('Nexar census', CENSUS), ('Nexar CLIP s1', NEXAR_X), ('Nexar V2-S s1', NEXAR_U), ('Nexar defs', NEXAR_DEFS),
          ('§13 LaGoVAD read', OLD / LAG / 'nexar_readout.json'), ('§13 K5 (T2) read', OLD / K5T2 / 'nexar_readout.json'))
missing = [f'{n}: {p}' for n, p in INPUTS if not p.exists()]
assert not missing, 'missing -- paste to Claude:\n' + '\n'.join(missing)
print(f'{len(INPUTS)} inputs found | output {NEW}')

In [ ]:
%%bash
pip install -q "transformers==4.56.*" av einops faiss-cpu scikit-learn
python -c "import av, einops, faiss, sklearn; print('deps OK')"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- step 4 needs one'

In [ ]:
import torch

from core.data.v2_splits import load_split      # noqa: E402

ENV = dict(os.environ, USE_TF='0', TF_CPP_MIN_LOG_LEVEL='2')   # transformers must not import TF


def run(cmd, log=None):
    '''Stream a child's output here (and to ``log``); raise with its tail on failure.'''
    tail = []
    sink = open(log, 'a') if log else None
    try:
        with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                tail = (tail + [line])[-40:]
                if sink:
                    sink.write(line)
                print(line, end='')
    finally:
        if sink:
            sink.close()
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')


def publish(src, dst):
    '''VM-local result -> Drive ATOMICALLY: ``<dst>.partial``, fsync, rename (C10).'''
    if src.is_dir():
        for f in [p for p in src.rglob('*') if p.is_file()]:
            publish(f, dst / f.relative_to(src))
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + '.partial')
    shutil.copy2(src, tmp)
    with open(tmp, 'rb+') as fh:
        os.fsync(fh.fileno())
    os.replace(tmp, dst)


def stage(src, dst, ids):
    '''``ids`` (+ json/npz manifests) of a Drive dir to VM disk once (bit-identical; C2).'''
    t0 = time.time()
    dst.mkdir(parents=True, exist_ok=True)
    for v in ids:
        if not (dst / f'{v}.npy').exists():
            shutil.copy2(src / f'{v}.npy', dst / f'{v}.npy')
    for pattern in ('*.json', '*.npz'):
        for m in src.glob(pattern):
            shutil.copy2(m, dst / m.name)
    print(f'  staged {src.name} in {time.time() - t0:.0f}s')
    return dst


def flush_drive():
    drive.flush_and_unmount()
    drive.mount('/content/drive')


def metrics_last(path):
    lines = [l for l in path.read_text().splitlines() if l.strip()] if path.exists() else []
    return json.loads(lines[-1]) if lines else None


def headline(r):
    '''The numbers §13 printed for one row, from a nexar_readout.json.'''
    return {'crops': r['crops']['mean_over_placements']['macro']['mean'],
            'whole': r['whole']['macro']['mean'], 'micro': r['whole']['micro'], 'ap': r['whole']['micro_ap']}


# Preflight on small files only (C39)
rec = json.loads((RUN / 'verified.json').read_text())
assert rec['step'] == STEPS and rec['ckpt_bytes'] == CKPT.stat().st_size, f'K5 (T2) not verified: {rec}'
assert metrics_last(RUN / 'metrics.jsonl')['global_step'] == STEPS, 'K5 (T2): J10'
man = json.loads((A2_T2 / constants.V2_INPUT_MANIFEST_FILENAME).read_text())
assert (man['crn'], man['motion']) == (constants.V2_OFF, ENC), f'T2 stats: {man}'
NEXAR_IDS = sorted(load_split(NEXAR_SPLIT))
for name, d in (('Nexar CLIP', NEXAR_X), ('Nexar V2-S', NEXAR_U)):
    miss = [v for v in NEXAR_IDS if not (d / f'{v}.npy').exists()]
    assert not miss, f'{len(miss)} of {name} missing: {miss[:5]}'
OLD_READS = {row: headline(json.loads((OLD / row / 'nexar_readout.json').read_text())) for row in (LAG, K5T2)}
for row, h in OLD_READS.items():
    print(f'§13 {row}: ' + ' | '.join(f'{k} {v:.4f}' for k, v in h.items()))
print(f'preflight OK: K5 (T2) step {STEPS} | {NEXAR_SPLIT} {len(NEXAR_IDS)} videos')

## 4. Score `nexar_val` again, now saving per-frame curves; check against §13

In [ ]:
from core.tools.nexar_eval import READOUT_JSON  # noqa: E402

assert torch.cuda.is_available(), 'torch sees no GPU -- nexar_eval would silently score on CPU'
L_X = stage(NEXAR_X, LOCAL / 'stage' / 'nexar_x', NEXAR_IDS)
L_U = stage(NEXAR_U, LOCAL / 'stage' / 'nexar_u', NEXAR_IDS)
L_DEFS = LOCAL / 'stage' / 'nexar_defs'
L_DEFS.mkdir(parents=True, exist_ok=True)
for f in NEXAR_DEFS.glob('*.json'):         # the definitions; the same for both rows
    shutil.copy2(f, L_DEFS / f.name)
L_T2_STATS = stage(A2_T2, LOCAL / 'stage' / 't2_stats', [])   # manifest + statistics only
ROWS = {
    LAG: (['--baseline', constants.ZS_LAGOVAD_RUN, BEST], []),   # first: the cheapest place to find a loader problem
    K5T2: (['--run', f's{SEED}', CKPT], ['--stats-dir', L_T2_STATS, '--video-s1-dir', L_U]),
}
for row, (ckpt_args, extra) in ROWS.items():
    out = NEW / row
    done = (out / READOUT_JSON).exists() and len(list((out / 'scores').glob('*.npz'))) == len(NEXAR_IDS)
    if not done:
        local = LOCAL / 'eval' / row
        shutil.rmtree(local, ignore_errors=True)
        run([sys.executable, '-m', 'core.tools.nexar_eval', *ckpt_args, *extra, '--census', CENSUS,
             '--clip-s1-dir', L_X, '--data-dir', L_DEFS, '--split', NEXAR_SPLIT, '--save-scores',
             '--out-dir', local], log=LOCAL / 'nexar.log')
        publish(local, out)
    r = json.loads((out / READOUT_JSON).read_text())
    assert r['split'] == NEXAR_SPLIT and r['videos'] == len(NEXAR_IDS), (r['split'], r['videos'])
    new = headline(r)
    gap = {k: abs(new[k] - OLD_READS[row][k]) for k in new}
    print(f'{row}: ' + ' | '.join(f'{k} {new[k]:.4f} (§13 {OLD_READS[row][k]:.4f})' for k in new))
    assert max(gap.values()) <= REPRO_TOL, f'{row} does not reproduce §13 -- paste to Claude: {gap}'
flush_drive()
print('both rows reproduce §13')

## 5. Draw: grid + onset-aligned mean, then one PNG per video (rank 1 = worst per-video AUC of K5 (T2))

In [ ]:
local = LOCAL / 'figures'
shutil.rmtree(local, ignore_errors=True)
reads = ['--read', NX, MAIN_LABEL, NEW / K5T2, '--read', NX, LAG, NEW / LAG]
common = ['--rank-by', MAIN_LABEL, '--name', TAG, '--out-dir', local, '--seed-label', f's{SEED}, n = 1']
run([sys.executable, '-m', 'core.tools.zs_curves', *common, *reads])            # grid_nexar.png + onset
run([sys.executable, '-m', 'core.tools.zs_curves', '--all', *common, *reads])   # videos/nexar/<rank>_<id>.png
publish(local, FIGS)
t = json.loads((local / f'{constants.ZS_CURVES_STEM}_{TAG}.json').read_text())['sets'][NX]
print(f"{t['videos']} videos, {t['two_class']} two-class | median per-video AUC: "
      + ', '.join(f'{m} {a:.3f}' for m, a in t['median_video_auc'].items()))
drive.flush_and_unmount()          # C10: every pending Drive upload lands before the VM goes
print(f'Drive flushed -- figures in {FIGS}')